In [ ]:
# Bibliotecas necessárias
import sys
import os

sys.path.append(
    os.path.abspath("../module")
)

import pandas as pd
import joblib

In [2]:
from feature_extractor import extract_features

In [ ]:
# Carregando o modelo Random Forest treinado para URL
rf_model = joblib.load(
    "../data/output_API/models/rf_model_url.pkl"
)

c:\Python\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Python\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [16]:
url_features = [

    'NumDots',
    'SubdomainLevel',
    'PathLevel',
    'UrlLength',
    'NumDash',
    'NumDashInHostname',
    'AtSymbol',
    'TildeSymbol',
    'NumUnderscore',
    'NumPercent',
    'NumQueryComponents',
    'NumAmpersand',
    'NumHash',
    'NumNumericChars',
    'NoHttps',
    'IpAddress',
    'DomainInSubdomains',
    'DomainInPaths',
    'HostnameLength',
    'PathLength',
    'QueryLength',
    'DoubleSlashInPath'
]

In [39]:
def predict_url(url):

    # Extração
    features = extract_features(url)

    # DataFrame
    df = pd.DataFrame([features])

    # Ordenando colunas
    df = df[url_features]

    # Probabilidades
    probabilities = rf_model.predict_proba(df)[0]

    legit_prob = probabilities[0]
    phishing_prob = probabilities[1]

    # =========================
    # Classificação de risco
    # =========================

    if phishing_prob < 0.30:

        risk = "Baixo Risco"
        prediction = "Legítima"

    elif phishing_prob < 0.60:

        risk = "Risco Moderado"
        prediction = "Suspeita"

    else:

        risk = "Alto Risco"
        prediction = "Phishing"

    # =========================
    # Retorno
    # =========================

    return {

        "url": url,

        "prediction": prediction,

        "legitimate_probability": float(
            round(legit_prob, 4) * 100 
        ),

        "phishing_probability": float(
            round(phishing_prob, 4) * 100
        ),

        "risk_level": risk
    }

In [40]:
result = predict_url(
    "http://paypal-login-security-update.com/verify?id=12345"
)

result

{'url': 'http://paypal-login-security-update.com/verify?id=12345',
 'prediction': 'Legítima',
 'legitimate_probability': 75.0,
 'phishing_probability': 25.0,
 'risk_level': 'Baixo Risco'}

In [41]:
result = predict_url(
    "http://paypal-login-secure.com/login?id=123"
)

result

{'url': 'http://paypal-login-secure.com/login?id=123',
 'prediction': 'Legítima',
 'legitimate_probability': 77.0,
 'phishing_probability': 23.0,
 'risk_level': 'Baixo Risco'}

In [42]:
result = predict_url(
    "http://paypal-login-security-update.com/verify/account?id=12345"
)

result

{'url': 'http://paypal-login-security-update.com/verify/account?id=12345',
 'prediction': 'Suspeita',
 'legitimate_probability': 66.0,
 'phishing_probability': 34.0,
 'risk_level': 'Risco Moderado'}

In [43]:
result = predict_url(
    "http://192.168.0.1/verify/login/update?id=99999"
)

result

{'url': 'http://192.168.0.1/verify/login/update?id=99999',
 'prediction': 'Suspeita',
 'legitimate_probability': 42.0,
 'phishing_probability': 57.99999999999999,
 'risk_level': 'Risco Moderado'}

In [44]:
result = predict_url(
    "http://255.255.255.255/paypal/secure/login/update?id=9999"
)

result

{'url': 'http://255.255.255.255/paypal/secure/login/update?id=9999',
 'prediction': 'Suspeita',
 'legitimate_probability': 53.0,
 'phishing_probability': 47.0,
 'risk_level': 'Risco Moderado'}